# Preprocessing — HPP sévère

Objectif : préparer les features **pré-accouchement** pour la modélisation.

| Décision | Choix retenu | Pourquoi |
|----------|--------------|----------|
| Valeurs manquantes | **`dropna`** | Meilleur recall / F1 qu’une imputation simple (expériences archivées) |
| Split | Stratifié 80/20 | Préserve le déséquilibre de classes |
| Encodage | `ColumnTransformer` | Quant / binaire / nominal / ordinal |

Expériences détaillées : `_archive/02_Modelisation_imputation.ipynb`.
Données complètes : `../00_Data/DATA_LOCATION.md`.

## 1. Chargement & typologie des features

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA = Path("..") / "00_Data"
TARGET = "hpp_trans"

# Aligné sur le modèle déployé (API / Streamlit)
QUANT = ["age_m", "taille_mere", "bmi", "parite", "hosp_m_g", "dsm_g", "nbilan", "nsej18", "terme"]
BINARY = ["tabac", "hta_tot", "cholestase", "hellp", "creta", "ut_cica", "cortico", "pma", "bilan"]
NOMINAL = ["diabete", "AMP"]
ORDINAL = ["preecl", "g_type"]
FEATURES = QUANT + BINARY + NOMINAL + ORDINAL

df = pd.read_csv(DATA / "extract_database.csv", low_memory=False)
cols = [c for c in FEATURES + [TARGET] if c in df.columns]
df = df[cols].copy()
print(f"Chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print("NA % (top) :")
print((df.isna().mean() * 100).sort_values(ascending=False).head(8).round(2))

## 2. Décision NA : dropna vs imputation

Sur la **base complète** (~60k lignes), une comparaison LogReg rapide a montré que `dropna`
surpasse une imputation médiane / most_frequent (meilleur F1 et recall).

| Stratégie | Intérêt | Limite déploiement |
|-----------|---------|---------------------|
| `dropna` (retenu) | Pas de biais d’imputation | Les nouvelles admissions doivent être **complètes** |
| Imputation | Conserve plus de lignes | Dilue le signal rare (~2 % positifs) |

Ci-dessous : illustration sur l’extrait portfolio (effectifs trop faibles pour rejouer le benchmark).

In [ ]:
n_before = len(df)
df_complete = df.dropna()
n_after = len(df_complete)

summary = pd.DataFrame({
    "stratégie": ["brut (extrait)", "après dropna"],
    "n_lignes": [n_before, n_after],
    "n_positifs": [
        int(df[TARGET].sum()) if TARGET in df.columns else None,
        int(df_complete[TARGET].sum()) if TARGET in df_complete.columns else None,
    ],
})
summary

## 3. Preprocessor sklearn

Même logique que Jedha ML : pipeline par type de variable, sans fuite d’info (fit sur train uniquement).

In [ ]:
def build_preprocessor(quant, binary, nominal, ordinal):
    """ColumnTransformer aligné sur le modèle en production."""
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),  # filet si NA résiduels
        ("scaler", StandardScaler()),
    ])
    nominal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric, quant),
        ("bin", "passthrough", binary),
        ("nom", nominal_pipe, nominal),
        ("ord", ordinal_pipe, ordinal),
    ])


available_quant = [c for c in QUANT if c in df_complete.columns]
available_bin = [c for c in BINARY if c in df_complete.columns]
available_nom = [c for c in NOMINAL if c in df_complete.columns]
available_ord = [c for c in ORDINAL if c in df_complete.columns]

preprocessor = build_preprocessor(available_quant, available_bin, available_nom, available_ord)
print("Preprocessor :", preprocessor)

## 4. Split stratifié (si données suffisantes)

Sur la base complète : `train_test_split(..., stratify=y, test_size=0.2, random_state=42)`.
L’extrait portfolio est trop petit / mono-classe pour un split réaliste — on vérifie juste la structure.

In [ ]:
feature_cols = available_quant + available_bin + available_nom + available_ord

if TARGET in df_complete.columns and df_complete[TARGET].nunique() >= 2 and len(df_complete) >= 50:
    X = df_complete[feature_cols]
    y = df_complete[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    preprocessor.fit(X_train)
    print(f"Train {X_train.shape} | Test {X_test.shape}")
    print("Positifs train %:", round(100 * y_train.mean(), 2))
else:
    print(
        "Extrait insuffisant pour un split stratifié réaliste.\n"
        "→ Relancer ce notebook avec la base complète (DATA_LOCATION.md).\n"
        "→ Les résultats métier sont documentés dans 03_Model_Comparison.ipynb."
    )
    print("Features retenues :", feature_cols)

## 5. Implications déploiement

- L’API / Streamlit attendent les **22 features** dans l’ordre de `feature_order.pkl`.
- Une ligne avec NA est rejetée en amont (cohérent avec `dropna`).
- Suite : `03_Model_Comparison.ipynb`.